# ML CA 1 — Hospital Readmissions: train · evaluate · submit

Single-notebook version of `main.py`, ready to run in **Google Colab** (also works locally).

**How to use:** `Runtime → Run all`. The dataset is downloaded automatically from the project's GitHub repo; if that fails (e.g. private repo), a manual upload prompt appears instead.

Fixes over the instructor's example notebook:
- `select_dtypes(...).column` → `.columns`
- the train/validation split unpacked values without calling `train_test_split`
- estimators passed as classes (`LinearRegression`) instead of instances
- categorical string features were imputed but never encoded, which crashes linear models — now one-hot encoded with `handle_unknown="ignore"`
- a yes/no target is a classification problem → LogisticRegression with accuracy / precision / recall / F1 / ROC-AUC (numeric targets keep LinearRegression with RMSE / MAE / R²)


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
print("pandas", pd.__version__)


In [ ]:
# ---- configuration: everything dataset-specific lives here ----
TRAIN_CSV = "hospital_readmissions.csv"
TEST_CSV = None            # set to the graded test csv name once released, e.g. "hospital_readmissions_test.csv"
TARGET = "readmitted"      # column to predict; None = use the last column of the train csv
ID_COLUMN = None           # None = auto-detect 'id' / '*_id', otherwise submissions are keyed by row index
TEST_SIZE = 0.40           # holdout fraction used for evaluation
MODEL = None               # None = auto: logistic (classification) or linear (regression); or "random_forest"
SUBMISSION_OUT = "submission.csv"

# Experiment only: predict on the training file itself. In-sample predictions are
# NOT a valid submission (roughly 39% of rows differ from the true labels).
DEMO_SUBMISSION_ON_TRAIN = False

DATA_URL_BASE = "https://raw.githubusercontent.com/zenpieV/mle_ca1_alok_29/main/"


In [ ]:
from pathlib import Path

def load_csv(name):
    # local file first, then the GitHub raw copy, then a manual upload prompt in Colab
    if Path(name).exists():
        return pd.read_csv(name)
    try:
        frame = pd.read_csv(DATA_URL_BASE + Path(name).name)
        print(f"Downloaded '{name}' from {DATA_URL_BASE}")
        return frame
    except Exception:
        pass
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"'{name}' not found locally and not downloadable from {DATA_URL_BASE}")
    print(f"Could not find or download '{name}' - please upload it:")
    uploaded = files.upload()
    return pd.read_csv(next(iter(uploaded)))


## Preprocessing and model

Numeric features: median imputation + standard scaling. Categorical features: most-frequent imputation + one-hot encoding (categories unseen at training time are ignored rather than crashing prediction). Everything is wrapped in a single sklearn `Pipeline`; the estimator is chosen automatically from the target's dtype.


In [ ]:
def detect_task(y):
    # text, boolean or categorical targets -> classification; numeric -> regression
    if (
        pd.api.types.is_bool_dtype(y)
        or pd.api.types.is_string_dtype(y)
        or pd.api.types.is_object_dtype(y)
        or isinstance(y.dtype, pd.CategoricalDtype)
    ):
        return "classification"
    return "regression"

def build_pipeline(task, model_name, X):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

    estimators = {
        ("classification", "logistic"): lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        ("classification", "random_forest"): lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        ("regression", "linear"): LinearRegression,
        ("regression", "random_forest"): lambda: RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    }
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimators[(task, model_name)]()),
    ])


In [ ]:
def evaluate(task, model, X_valid, y_valid):
    predictions = model.predict(X_valid)

    if task == "classification":
        classes = list(model.named_steps["model"].classes_)
        if len(classes) == 2:
            average, pos_label = "binary", classes[-1]
        else:
            average, pos_label = "weighted", None
        metrics = {
            "accuracy": accuracy_score(y_valid, predictions),
            "precision": precision_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
            "recall": recall_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
            "f1": f1_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
        }
        probabilities = model.predict_proba(X_valid)
        if len(classes) == 2:
            y_true_binary = (pd.Series(y_valid) == pos_label).astype(int)
            metrics["roc_auc"] = roc_auc_score(y_true_binary, probabilities[:, classes.index(pos_label)])
        else:
            metrics["roc_auc"] = roc_auc_score(y_valid, probabilities, multi_class="ovr", average="weighted", labels=classes)
    else:
        metrics = {
            "rmse": float(np.sqrt(mean_squared_error(y_valid, predictions))),
            "mae": mean_absolute_error(y_valid, predictions),
            "r2": r2_score(y_valid, predictions),
        }
    return metrics


## Load data and set up the task


In [ ]:
train = load_csv(TRAIN_CSV)

target = TARGET or train.columns[-1]
if target not in train.columns:
    raise ValueError(f"Target column '{target}' not found. Available: {list(train.columns)}")

id_column = ID_COLUMN
if id_column is None:
    candidates = [c for c in train.columns if c != target and re.match(r"(?:^|_)id$", c, flags=re.IGNORECASE)]
    id_column = candidates[0] if candidates else None

drop_columns = [target] + ([id_column] if id_column in train.columns else [])
X = train.drop(columns=drop_columns)
y = train[target]

task = detect_task(y)
model_name = MODEL or ("logistic" if task == "classification" else "linear")

VALID_COMBINATIONS = {
    ("classification", "logistic"),
    ("classification", "random_forest"),
    ("regression", "linear"),
    ("regression", "random_forest"),
}
if (task, model_name) not in VALID_COMBINATIONS:
    raise ValueError(f"Model '{model_name}' cannot be used for a {task} target.")

print(f"Train data : {train.shape[0]} rows x {train.shape[1]} columns")
print(f"Target     : '{target}' ({task})")
print(f"ID column  : {id_column if id_column else 'none found - submissions will use row index'}")
print(f"Model      : {model_name}")
if task == "classification":
    print(f"Classes    : {y.value_counts().to_dict()}")


## Train and evaluate on a holdout split


In [ ]:
stratify = y if task == "classification" and y.value_counts().min() > 1 else None
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
)

model = build_pipeline(task, model_name, X)
model.fit(X_train, y_train)
metrics = evaluate(task, model, X_valid, y_valid)

print(f"Evaluation on {TEST_SIZE:.0%} holdout ({len(X_valid)} rows):")
for name, value in metrics.items():
    print(f"  {name:<9} {value:.4f}")


## Submission

`TEST_CSV` is currently `None`, so this cell reports evaluation results only. When the graded test csv is released: put its name in `TEST_CSV` in the configuration cell and re-run — the model is refit on all labelled rows and predictions are written to `submission.csv` (offered as a download in Colab).


In [ ]:
if TEST_CSV:
    test = load_csv(TEST_CSV)
    if target in test.columns:
        print(f"Note: test csv already contains '{target}' - dropping it before predicting.")
        test = test.drop(columns=[target])

    X_test = test[[c for c in test.columns if c != id_column]]
    final_model = build_pipeline(task, model_name, X)
    final_model.fit(X, y)
    test_predictions = final_model.predict(X_test)

    if id_column and id_column in test.columns:
        submission = pd.DataFrame({id_column: test[id_column], target: test_predictions})
    else:
        submission = pd.DataFrame({"row_id": test.index, target: test_predictions})
    submission.to_csv(SUBMISSION_OUT, index=False)
    print(f"Final model refit on all {len(X)} labelled rows.")
    print(f"Wrote {SUBMISSION_OUT} ({len(submission)} predictions, columns: {list(submission.columns)})")

    try:
        from google.colab import files
        files.download(SUBMISSION_OUT)
    except ImportError:
        pass
elif DEMO_SUBMISSION_ON_TRAIN:
    # experiment only - in-sample predictions on the training file, NOT a valid submission
    demo_model = build_pipeline(task, model_name, X)
    demo_model.fit(X, y)
    demo = pd.DataFrame({"row_id": train.index, target: demo_model.predict(X)})
    demo.to_csv(SUBMISSION_OUT, index=False)
    print(f"Wrote {SUBMISSION_OUT} from DEMO in-sample predictions ({len(demo)} rows) - do not submit this.")
else:
    print("No test set configured - evaluation only. Set TEST_CSV in the configuration cell and re-run to write", SUBMISSION_OUT)
